# recount2 — C2CP Pathway Coverage vs Sample Size (scaled multiplier + max.iter)

**Environment:** `clamp-analyses`

Plots the C2CP pathway coverage and K results produced by `05_recount2_coverage_multiplier_imp.ipynb`.

Coverage = proportion of input C2CP pathways significantly associated with an LV (FDR < 0.05),
following the `GetPathwayCoverage` approach from the multi-plier paper (Taroni 2018).

Input: `output/recount2_multiplier_imp/c2cp_subsample_{N}_seed_{seed}/CLAMPfull_C2CP.rds`

Seeds and CLAMP_K per sample size are read from `data/archs4/subsampled_number_of_lvs.tsv`.

`multiplier` and `max.iter` used per run are read from the saved `subsample_info.rds`.

## Libraries

In [ ]:
library(dplyr)
library(ggplot2)
library(here)

source(here("config.R"))

## Configuration

In [ ]:
FDR_CUTOFF <- 0.05
output_dir <- file.path(config$GENERAL$OUTPUT_DIR, "recount2_multiplier_imp")

lvs_tsv <- read.table(
    here("data", "archs4", "subsampled_number_of_lvs.tsv"),
    header     = TRUE,
    sep        = "\t",
    colClasses = c(number_of_latent_variables = "integer",
                   seed                       = "integer",
                   sample_size                = "integer")
)

# Sort by sample_size ascending so rows are processed 500 → 32000
lvs_tsv <- lvs_tsv[order(lvs_tsv$sample_size, lvs_tsv$seed), ]
rownames(lvs_tsv) <- NULL

## Coverage function

In [ ]:
# Adapted from GetPathwayCoverage (Taroni 2018 / multi-plier)
GetPathwayCoverage <- function(clamp.result, fdr.cutoff = 0.05) {
    summary.df     <- clamp.result$summary
    input.pathways <- colnames(clamp.result$C)
    num.lvs        <- ncol(clamp.result$U)

    sig.pathways <- unique(summary.df$pathway[which(summary.df$FDR < fdr.cutoff)])
    sig.lvs      <- unique(summary.df$LV[which(summary.df$FDR < fdr.cutoff)])

    list(
        pathway           = length(sig.pathways) / length(input.pathways),
        lv                = length(sig.lvs) / num.lvs,
        sig.pathway.by.lv = length(sig.pathways) / num.lvs
    )
}

## Collect results per run

In [ ]:
rows <- list()

for (i in seq_len(nrow(lvs_tsv))) {
    n_target     <- lvs_tsv$sample_size[i]
    current_seed <- lvs_tsv$seed[i]
    clamp_k_tsv  <- lvs_tsv$number_of_latent_variables[i]

    run_dir  <- file.path(output_dir,
                          paste0("c2cp_subsample_", n_target, "_seed_", current_seed))
    rds_path <- file.path(run_dir, "CLAMPfull_C2CP.rds")
    k_path   <- file.path(run_dir, "CLAMP_K.rds")
    info_path <- file.path(run_dir, "subsample_info.rds")

    if (!file.exists(rds_path)) {
        message("Not found — skipping: c2cp_subsample_",
                n_target, "_seed_", current_seed)
        next
    }

    model   <- readRDS(rds_path)
    clamp_k <- if (file.exists(k_path)) readRDS(k_path) else clamp_k_tsv
    cov     <- GetPathwayCoverage(model, fdr.cutoff = FDR_CUTOFF)

    # Read multiplier and max_iter from saved subsample_info if available
    multiplier <- NA_real_
    max_iter   <- NA_integer_
    if (file.exists(info_path)) {
        info       <- readRDS(info_path)
        multiplier <- info$multiplier
        max_iter   <- info$max_iter
    }

    rows[[length(rows) + 1]] <- data.frame(
        sample_size      = n_target,
        n_samples        = ncol(model$B),
        seed             = current_seed,
        pathway_coverage = cov$pathway,
        lv_coverage      = cov$lv,
        clamp_k          = clamp_k,
        multiplier       = multiplier,
        max_iter         = max_iter
    )
    message(sprintf("n=%6d  seed=%d  pathway_cov=%.2f%%  K=%d  multiplier=%g  max.iter=%d",
                    n_target, current_seed,
                    cov$pathway * 100, clamp_k,
                    multiplier, max_iter))
}

results_df <- dplyr::bind_rows(rows) %>%
    dplyr::mutate(sample_size = factor(sample_size,
                                       levels = sort(unique(sample_size))))
print(results_df)

## Pathway coverage

In [ ]:
p_cov <- ggplot(results_df,
                aes(x = sample_size, y = pathway_coverage * 100)) +
    geom_boxplot(fill = "#FF9800", color = "#E65100",
                 alpha = 0.6, outlier.shape = NA, width = 0.5) +
    geom_jitter(color = "#E65100", width = 0.1, size = 2.5, alpha = 0.8) +
    labs(
        x = "Number of samples",
        y = "Pathway coverage (%)"
    ) +
    theme_bw(base_size = 13) +
    theme(
        axis.text.x      = element_text(angle = 45, hjust = 1),
        panel.grid.minor = element_blank()
    )

print(p_cov)

## K (number of components)

In [ ]:
p_k <- ggplot(results_df,
              aes(x = sample_size, y = clamp_k)) +
    geom_boxplot(fill = "#FF9800", color = "#E65100",
                 alpha = 0.6, outlier.shape = NA, width = 0.5) +
    geom_jitter(color = "#E65100", width = 0.1, size = 2.5, alpha = 0.8) +
    labs(
        x = "Number of samples",
        y = "K (number of components)"
    ) +
    theme_bw(base_size = 13) +
    theme(
        axis.text.x      = element_text(angle = 45, hjust = 1),
        panel.grid.minor = element_blank()
    )

print(p_k)

## Multiplier used per sample size

In [ ]:
p_mult <- ggplot(results_df,
                 aes(x = sample_size, y = multiplier)) +
    geom_boxplot(fill = "#29B6F6", color = "#0277BD",
                 alpha = 0.6, outlier.shape = NA, width = 0.5) +
    geom_jitter(color = "#0277BD", width = 0.1, size = 2.5, alpha = 0.8) +
    labs(
        x = "Number of samples",
        y = "CLAMPfull multiplier"
    ) +
    theme_bw(base_size = 13) +
    theme(
        axis.text.x      = element_text(angle = 45, hjust = 1),
        panel.grid.minor = element_blank()
    )

print(p_mult)

## Summary table

In [ ]:
summary_tbl <- results_df %>%
    dplyr::group_by(sample_size) %>%
    dplyr::summarise(
        median_cov = round(median(pathway_coverage) * 100, 2),
        min_cov    = round(min(pathway_coverage) * 100, 2),
        max_cov    = round(max(pathway_coverage) * 100, 2),
        median_k   = median(clamp_k, na.rm = TRUE),
        multiplier = unique(multiplier),
        max_iter   = unique(max_iter),
        n          = dplyr::n(),
        .groups    = "drop"
    ) %>%
    dplyr::arrange(sample_size)

print(summary_tbl)

write.csv(
    summary_tbl,
    file.path(output_dir, "subsample_results_summary_multiplier_imp_coverage.csv"),
    row.names = FALSE
)